# Audio chunking for DAIC-WOZ using csv timestamps

In [ ]:
import os
import csv
from pydub import AudioSegment
from pydub.silence import split_on_silence


RAW_AUDIO_DIR = r"D:\thesis_data\DAIC-WOZ\split_data\gated_audio"
CSV_DIR = r"D:\thesis_data\DAIC-WOZ\split_data\csv_transcripts_daic"
OUTPUT_DIR = r"D:\thesis_data\DAIC-WOZ\mfa_corpus_daic_NEW"
LOG_CSV = "segments_log_DAIC.csv"

MIN_START_TIME = 120.0      # only extract audio chunks after 2 minutes
MIN_CHUNK_DURATION = 3.0    # minimum final chunk duration (seconds)
MAX_CHUNK_DURATION = 25.0   # stop transcript merging at this length
MAX_GAP = 2.0               # max allowed gap between transcript segments
PADDING = 0.5               # pad chunk end slightly (seconds)

os.makedirs(OUTPUT_DIR, exist_ok=True)

log_rows = []


def process_participant(participant_id):
    wav_path = os.path.join(RAW_AUDIO_DIR, f"{participant_id}_AUDIO.wav")
    csv_path = os.path.join(CSV_DIR, f"{participant_id}_Transcript.csv")

    if not os.path.exists(wav_path) or not os.path.exists(csv_path):
        print(f"Skipping {participant_id}, missing files")
        return

    audio = AudioSegment.from_wav(wav_path)

    # Load transcript segments
    segments = []
    with open(csv_path, newline='', encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            start = float(row["Start_Time"])
            end = float(row["End_Time"])
            text = row["Text"].strip()

            if start < MIN_START_TIME or text == "":
                continue

            segments.append((start, end, text))

    chunk_id = 1
    i = 0

    while i < len(segments):
        start, end, text = segments[i]
        current_start = start
        current_end = end
        texts = [text]

        j = i + 1
        while j < len(segments):
            next_start, next_end, next_text = segments[j]
            gap = next_start - current_end

            if gap > MAX_GAP:
                break

            current_end = next_end
            texts.append(next_text)
            j += 1

            duration = current_end - current_start
            if duration >= MAX_CHUNK_DURATION:
                break

        duration = current_end - current_start

        num_words = sum(len(t.split()) for t in texts)
        status = "Used" if duration >= MIN_CHUNK_DURATION else "Ignored"

        log_rows.append({
            "Participant_ID": participant_id,
            "Segment_ID": f"{chunk_id:03d}",
            "Num_Words": num_words,
            "Status": status
        })

        if status == "Used":
            chunk_end_ms = int(min(current_end + PADDING, len(audio) / 1000) * 1000)
            chunk_audio = audio[int(current_start * 1000):chunk_end_ms]

            
            if duration >= 30.0:
                sub_chunks = split_on_silence(
                    chunk_audio,
                    min_silence_len=2000,                # 2 seconds
                    silence_thresh=chunk_audio.dBFS - 16,
                    keep_silence=500
                )
            else:
                sub_chunks = [chunk_audio]

            for sub_chunk in sub_chunks:
                sub_duration = len(sub_chunk) / 1000.0

                if sub_duration < MIN_CHUNK_DURATION:
                    continue

                base_name = f"{participant_id}_{chunk_id:03d}"
                wav_out = os.path.join(OUTPUT_DIR, base_name + ".wav")

                sub_chunk.export(wav_out, format="wav")

                chunk_id += 1

        i = j

    print(f"Processed {participant_id}")


# Run for all participants
for file in os.listdir(RAW_AUDIO_DIR):
    if file.endswith("_AUDIO.wav"):
        participant_id = file.replace("_AUDIO.wav", "")
        process_participant(participant_id)


# Save log CSV
with open(LOG_CSV, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["Participant_ID", "Segment_ID", "Num_Words", "Status"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in log_rows:
        writer.writerow(row)

print(f"Done building MFA corpus. Log saved to {LOG_CSV}")
